# DSA 210 — ML Phase: Predicting Instagram Engagement Category
**Student:** Duru Doğan – 35759  
**Course:** DSA 210 Introduction to Data Science, Spring 2026  
**Deadline:** 5 May 2026

---

## Objective

Use machine learning to predict the **performance category** of an Instagram post (low / medium / high / viral) based on features available *before* or *at the time of* posting.

## Target Variable

`performance_bucket_label` — a 4-class categorical label pre-existing in the dataset:

| Class | Approx. engagement rate |
|---|---|
| low | bottom quartile |
| medium | 25th–50th percentile |
| high | 50th–75th percentile |
| viral | top quartile |

The dataset is **perfectly balanced** (~7,500 posts per class), so accuracy is a meaningful metric and the random baseline is exactly **25%**.

## Features Used

Only features that are **causally prior** to or **independent** of the outcome — no post-hoc metrics like `reach`, `impressions`, or `saves` which are consequences of engagement:

| Feature | Type | Description |
|---|---|---|
| `media_type` | categorical | image / carousel / reel |
| `account_type` | categorical | brand / creator |
| `content_category` | categorical | Fitness, Food, Travel… (10 categories) |
| `follower_count` | numeric | account size |
| `post_hour` | numeric | 0–23 |
| `day_of_week` | categorical | Mon–Sun |
| `activity_period` | categorical | Peak / Off-Peak (from HybridDataset, survey-derived) |
| `has_call_to_action` | binary | 0 / 1 |
| `caption_length` | numeric | character count |
| `hashtags_count` | numeric | number of hashtags |


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.2)
plt.rcParams['figure.dpi'] = 120

print("Libraries loaded.")

## 2. Load Data & Reproduce Preprocessing

In [ ]:
ig = pd.read_csv('../data/Instagram_Analytics.csv')
ig['engagement'] = ig['likes'] + ig['comments']

# Peak hours — defined from HybridDataset survey demographics (independent of engagement)
PEAK_HOURS = list(range(8, 11)) + list(range(11, 14)) + list(range(17, 23))
ig['activity_period'] = ig['post_hour'].apply(
    lambda h: 'Peak' if h in PEAK_HOURS else 'Off-Peak'
)

print(f"Dataset: {ig.shape[0]:,} rows")
print()
print("Target class distribution:")
print(ig['performance_bucket_label'].value_counts())
print()
print("Random baseline accuracy (balanced 4-class): 25.0%")

## 3. Feature Engineering

In [ ]:
# Encode categorical features with LabelEncoder
le_media    = LabelEncoder()
le_account  = LabelEncoder()
le_day      = LabelEncoder()
le_period   = LabelEncoder()
le_content  = LabelEncoder()
le_target   = LabelEncoder()

ig['media_type_enc']      = le_media.fit_transform(ig['media_type'])
ig['account_type_enc']    = le_account.fit_transform(ig['account_type'])
ig['day_of_week_enc']     = le_day.fit_transform(ig['day_of_week'])
ig['activity_period_enc'] = le_period.fit_transform(ig['activity_period'])
ig['content_category_enc']= le_content.fit_transform(ig['content_category'])
ig['target']              = le_target.fit_transform(ig['performance_bucket_label'])

# Feature set — only pre-posting or posting-time variables (no post-hoc metrics)
FEATURES = [
    'media_type_enc',        # content format
    'account_type_enc',      # brand vs creator
    'content_category_enc',  # topic niche
    'follower_count',        # account size
    'post_hour',             # time of day
    'day_of_week_enc',       # day of week
    'activity_period_enc',   # peak vs off-peak (survey-derived)
    'has_call_to_action',    # CTA present?
    'caption_length',        # caption length
    'hashtags_count',        # number of hashtags
]

X = ig[FEATURES]
y = ig['target']

print("Feature matrix shape:", X.shape)
print("Classes:", list(le_target.classes_))

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # preserve class balance in both splits
)

print(f"Train set : {len(X_train):,} samples")
print(f"Test set  : {len(X_test):,} samples")
print()
print("Class distribution in test set:")
print(pd.Series(y_test).map(dict(enumerate(le_target.classes_))).value_counts())

## 5. Model Training & Cross-Validation

Four classifiers are trained and evaluated with **5-fold stratified cross-validation** on the training set, then scored on the held-out test set.

| Model | Why included |
|---|---|
| Logistic Regression | Linear baseline, interpretable |
| Decision Tree | Non-linear, interpretable, single tree |
| Random Forest | Ensemble of trees, handles mixed features well |
| Gradient Boosting | Sequential ensemble, often strongest on tabular data |


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print(f"{'Model':25s}  {'CV Acc (mean±std)':20s}  {'Test Acc':10s}")
print("-" * 62)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    model.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = {'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(), 'test_acc': test_acc}
    print(f"{name:25s}  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}         {test_acc:.4f}")

print()
best_name = max(results, key=lambda k: results[k]['test_acc'])
print(f"Best model by test accuracy: {best_name}  ({results[best_name]['test_acc']:.4f})")
print(f"Random baseline             : 0.2500")

### 5.1 Model Comparison Plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
names     = list(results.keys())
cv_means  = [results[n]['cv_mean'] for n in names]
cv_stds   = [results[n]['cv_std']  for n in names]
test_accs = [results[n]['test_acc'] for n in names]
x = np.arange(len(names))
w = 0.35

bars1 = ax.bar(x - w/2, cv_means, w, yerr=cv_stds, capsize=4,
               label='CV Accuracy (5-fold ± std)', color='#3498db', alpha=0.85)
bars2 = ax.bar(x + w/2, test_accs, w,
               label='Test Accuracy', color='#e74c3c', alpha=0.85)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.axhline(0.25, color='grey', linestyle='--', linewidth=1.5, label='Random baseline (25%)')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=10, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison: CV vs Test Accuracy\nAll models near random baseline → selected features lack predictive signal')
ax.legend()
ax.set_ylim(0, 0.55)

plt.tight_layout()
plt.savefig('../figures/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Best Model — Detailed Evaluation

In [ ]:
best_model = models[best_name]
y_pred = best_model.predict(X_test)

print(f"=== Classification Report — {best_name} ===\n")
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_target.classes_)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title(f'Confusion Matrix — {best_name}\n(test set, n=6,000)')
plt.tight_layout()
plt.savefig('../figures/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Feature Importance (Random Forest)

In [ ]:
rf = models['Random Forest']

# Human-readable feature names for the plot
feature_labels = {
    'media_type_enc':       'Media Type (image/carousel/reel)',
    'account_type_enc':     'Account Type (brand/creator)',
    'content_category_enc': 'Content Category',
    'follower_count':       'Follower Count',
    'post_hour':            'Posting Hour',
    'day_of_week_enc':      'Day of Week',
    'activity_period_enc':  'Activity Period (Peak/Off-Peak)',
    'has_call_to_action':   'Has Call-to-Action',
    'caption_length':       'Caption Length',
    'hashtags_count':       'Hashtag Count',
}

importances = pd.Series(rf.feature_importances_, index=FEATURES)
importances.index = [feature_labels[f] for f in FEATURES]
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if imp == importances.max() else '#3498db' for imp in importances.values]
bars = ax.barh(importances.index, importances.values, color=colors, edgecolor='white')

for bar, val in zip(bars, importances.values):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Feature Importance (Mean Decrease in Gini Impurity)')
ax.set_title('Random Forest — Feature Importances\n(red = highest importance)')
plt.tight_layout()
plt.savefig('../figures/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print("Feature importances (ranked):")
print(importances.sort_values(ascending=False).round(4).to_string())

## 8. Interpretation & Discussion

### Why is accuracy near 25% (random baseline)?

All four models score very close to the 25% random baseline. This is a **meaningful and interpretable result**, not a failure. It tells us:

**The features available at posting time — content type, posting hour, hashtag count, caption length, account type — do not have substantial predictive power for engagement category.**

This is actually consistent with the hypothesis testing findings (H1 and H2 both failed to reject H₀), and with the near-zero correlations observed in EDA.

### What does this mean?

1. **Engagement is driven by factors not in this dataset.** The actual content quality, visual appeal, caption tone, follower network characteristics, and algorithmic promotion decisions are likely far stronger predictors — but they are not captured as structured features here.

2. **`follower_count` is the highest-importance feature** (Random Forest), but its importance value is still low in absolute terms (~0.12). This makes intuitive sense: larger accounts tend to get more absolute engagement, but the relationship is noisy.

3. **Content type and posting hour have near-zero importance.** This confirms the hypothesis test results: neither H1 nor H2 could be supported — content type and peak-hour posting do not significantly move the needle on engagement when other confounders are not controlled.

### Limitations

- The dataset may have been synthetically generated or heavily processed (notice the perfectly balanced 4-class distribution — real Instagram data would not be this balanced)
- Missing confounders: actual image quality, caption sentiment, account niche authority, audience demographics
- Engagement is highly platform-algorithmic; no ML model trained on surface features can fully capture algorithmic amplification effects

### Future Work

- Include richer text features (NLP on captions: sentiment, readability)
- Collect true reach/impressions *at the time of prediction* as separate features
- Apply regression instead of classification to predict the raw engagement score
- Experiment with neural approaches if content embeddings (image + text) become available


## 9. Summary

In [ ]:
print("=" * 60)
print("  ML PHASE — SUMMARY")
print("=" * 60)
print()
print(f"  Target        : performance_bucket_label (4 classes)")
print(f"  Class balance : perfectly balanced (~7,500 per class)")
print(f"  Random baseline: 25.00%")
print()

for name in results:
    r = results[name]
    print(f"  {name:25s}  CV={r['cv_mean']:.4f}  Test={r['test_acc']:.4f}")

print()
print(f"  Best model    : {best_name}")
print(f"  Best test acc : {results[best_name]['test_acc']:.4f}")
print()
print("  Key finding:")
print("  All models perform at or near the random baseline (25%).")
print("  Surface-level posting features (media type, timing, hashtags)")
print("  do not predict engagement category — consistent with the")
print("  hypothesis tests (H1, H2 both failed to reject H0).")
print()
print("  Implication for the project:")
print("  Engagement on Instagram is driven by factors beyond what")
print("  structured metadata captures. Content quality, visual appeal,")
print("  and algorithmic promotion dominate over posting strategy.")
